In [2]:

"""
Batch processing script for MAV blob analysis.

This script:
1. Finds all .tif files in a directory and its subdirectories
2. Processes each file with process_image_pipeline
3. Combines all results into a single DataFrame
4. Exports the combined results to Excel
"""

import os
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import warnings



In [3]:


# Add current directory to Python path to import process_image
sys.path.append(".")

from process_image import process_image_pipeline
from skimage import io as skio

def find_tif_files(directory, recursive=True):
    """
    Find all .tif files in a directory.
    
    Args:
        directory: Path to search for TIF files
        recursive: If True, search subdirectories too
        
    Returns:
        List of Path objects for all found TIF files
    """
    directory = Path(directory)
    if not directory.exists():
        raise ValueError(f"Directory does not exist: {directory}")
    
    if recursive:
        return list(directory.rglob('*.tif')) + list(directory.rglob('*.tiff'))
    else:
        return list(directory.glob('*.tif')) + list(directory.glob('*.tiff'))

def process_single_file(file_path, min_blob_size=10, max_blob_size=1000, n_steps=10, save_images=False):
    """
    Process a single TIF file and return results DataFrame.
    
    Args:
        file_path: Path to TIF file
        min_blob_size: Minimum blob area in pixels
        max_blob_size: Maximum blob area in pixels
        n_steps: Number of expansion steps for profiles
        save_images: Whether to save visualization images
        
    Returns:
        DataFrame with analysis results for this file
    """
    try:
        # Load image data
        image_data = skio.imread(file_path)
        
        # Process with pipeline
        result_df = process_image_pipeline(
            image_data,
            filename=file_path.name,
            min_blob_size=min_blob_size,
            max_blob_size=max_blob_size,
            n_steps=n_steps,
            save_images=save_images
        )
        
        # Add file path information
        if len(result_df) > 0:
            result_df['file_path'] = str(file_path)
            try:
                # Try to get relative path, fall back to absolute if not possible
                result_df['relative_path'] = str(file_path.relative_to(Path.cwd()))
            except ValueError:
                # File is not in subpath of current directory
                result_df['relative_path'] = str(file_path)
        
        return result_df
        
    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")
        return pd.DataFrame()  # Return empty DataFrame on error

def batch_process_directory(directory, output_excel='mav_analysis_results.xlsx', 
                          min_blob_size=10, max_blob_size=1000, n_steps=10, 
                          save_images=False, recursive=True):
    """
    Batch process all TIF files in a directory.
    
    Args:
        directory: Directory containing TIF files
        output_excel: Output Excel file name
        min_blob_size: Minimum blob area in pixels
        max_blob_size: Maximum blob area in pixels
        n_steps: Number of expansion steps for profiles
        save_images: Whether to save visualization images
        recursive: Whether to search subdirectories
        
    Returns:
        Combined DataFrame with all results
    """
    print(f"🔍 Searching for TIF files in: {directory}")
    
    # Find all TIF files
    tif_files = find_tif_files(directory, recursive=recursive)
    
    if not tif_files:
        print("⚠️  No TIF files found!")
        return pd.DataFrame()
    
    print(f"📁 Found {len(tif_files)} TIF files to process")
    
    # Process files one by one
    all_results = []
    
    for file_path in tqdm(tif_files, desc="Processing files"):
        try:
            result_df = process_single_file(
                file_path,
                min_blob_size=min_blob_size,
                max_blob_size=max_blob_size,
                n_steps=n_steps,
                save_images=save_images
            )
            
            if len(result_df) > 0:
                all_results.append(result_df)
                print(f"✅ Processed {file_path.name}: {len(result_df)} blobs found")
            else:
                print(f"⚠️  Processed {file_path.name}: 0 blobs found")
                
        except Exception as e:
            print(f"❌ Error processing {file_path}: {e}")
    
    # Combine all results
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        print(f"📊 Combined results: {len(combined_df)} total blobs from {len(all_results)} files")
        
        # Add summary statistics
        print(f"📈 Summary statistics:")
        print(f"   - Files processed: {len(tif_files)}")
        print(f"   - Files with blobs: {len(all_results)}")
        print(f"   - Total blobs found: {len(combined_df)}")
        print(f"   - Mean blobs per file: {len(combined_df) / len(all_results):.1f}")
        
        # Save to Excel (fall back to CSV if openpyxl not available)
        try:
            # Try to import openpyxl
            import openpyxl
            
            # Create multiple sheets
            with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
                # Main data sheet
                combined_df.to_excel(writer, sheet_name='All_Blobs', index=False)
                
                # Summary statistics sheet
                summary_stats = {
                    'Total Files Processed': [len(tif_files)],
                    'Files with Blobs': [len(all_results)],
                    'Total Blobs Found': [len(combined_df)],
                    'Mean Blobs per File': [len(combined_df) / len(all_results) if all_results else 0],
                    'Processing Date': [pd.Timestamp.now()]
                }
                pd.DataFrame(summary_stats).to_excel(writer, sheet_name='Summary')
                
                # Statistics by file
                if 'filename' in combined_df.columns:
                    file_stats = combined_df.groupby('filename').agg({
                        'blob_index': 'count',
                        'blob_area': ['mean', 'std', 'min', 'max']
                    })
                    file_stats.columns = ['_'.join(col).strip() for col in file_stats.columns.values]
                    file_stats.to_excel(writer, sheet_name='File_Statistics')
            
            print(f"💾 Results saved to Excel: {output_excel}")
            
        except ImportError:
            # Fall back to CSV if openpyxl not available
            csv_output = output_excel.replace('.xlsx', '.csv')
            combined_df.to_csv(csv_output, index=False)
            print(f"💾 Results saved to CSV (openpyxl not available): {csv_output}")
            
        except Exception as e:
            print(f"❌ Error saving results: {e}")
            # Try to save as CSV as fallback
            try:
                csv_output = output_excel.replace('.xlsx', '.csv')
                combined_df.to_csv(csv_output, index=False)
                print(f"💾 Results saved to CSV fallback: {csv_output}")
            except Exception as e2:
                print(f"❌ Error saving CSV fallback: {e2}")
            
        return combined_df
    else:
        print("⚠️  No valid results to combine")
        return pd.DataFrame()

In [7]:
directory = "In"
output = "Out"
recursive = False
save_images = True
min_size = 5
max_size = 1000
profile_steps = 10
    
# Run batch processing
result_df = batch_process_directory(
    directory=directory,
    output_excel=output,
    min_blob_size=min_size,
    max_blob_size=max_size,
    n_steps=profile_steps,
    save_images=save_images,
    recursive=recursive
)

print("✅ Processing complete!")



🔍 Searching for TIF files in: In
📁 Found 2 TIF files to process


Processing files:   0%|                                                                                                                                        | 0/2 [00:00<?, ?it/s]

Successfully identified and traced 266 blobs.
Cleaned image contains 266 blobs.
Generating 10-step area-normalized profiles for 266 blobs...
Profile Matrix Generation Complete.
Shape: (266, 10) (Row: Blob, Column: Distance from Trace)
Saved overview image to: Out/Overview/ULK101_1 uM_4.jpg


Processing files:  50%|████████████████████████████████████████████████████████████████                                                                | 1/2 [01:30<01:30, 90.43s/it]

Saved 266 blob images to: Out/Clusters/ULK101_1 uM_4
✅ Processed ULK101_1 uM_4.tif: 266 blobs found
Successfully identified and traced 266 blobs.
Cleaned image contains 266 blobs.
Generating 10-step area-normalized profiles for 266 blobs...
Profile Matrix Generation Complete.
Shape: (266, 10) (Row: Blob, Column: Distance from Trace)
Saved overview image to: Out/Overview/ULK101_1 uM_3.jpg


Processing files: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [03:04<00:00, 92.32s/it]

Saved 266 blob images to: Out/Clusters/ULK101_1 uM_3
✅ Processed ULK101_1 uM_3.tif: 266 blobs found
📊 Combined results: 532 total blobs from 2 files
📈 Summary statistics:
   - Files processed: 2
   - Files with blobs: 2
   - Total blobs found: 532
   - Mean blobs per file: 266.0


IsADirectoryError: [Errno 21] Is a directory: 'Out'

In [ ]:
result_df